# Terminations in Microsoft Autogen

## Why Termination Matters

We have used team to do some work, say write a story or accomplish some task. 

In [1]:
import asyncio
from autogen_ext.models.openai import OpenAIChatCompletionClient
from dotenv import load_dotenv
import os

load_dotenv()
api_key = os.getenv('OPENAI_API_KEY')
model_client = OpenAIChatCompletionClient(model='gpt-4o', api_key=api_key)

In [2]:
from autogen_agentchat.agents import AssistantAgent
add_1_agent_first = AssistantAgent(
    name = 'add_1_agent_first',
    model_client=model_client,
    system_message="Add 1 to the number, first number is 0. Give result as output"
)

add_1_agent_second = AssistantAgent(
    name = 'add_1_agent_second',
    model_client=model_client,
    system_message="Add 1 to the number. Give result as output."
)
 
add_1_agent_third = AssistantAgent(
    name = 'add_1_agent_third',
    model_client=model_client,
    system_message="Add 1 to the number. Give result as output."
)

from autogen_agentchat.teams import RoundRobinGroupChat

team = RoundRobinGroupChat(
    [add_1_agent_first, add_1_agent_second, add_1_agent_third]
)

In [3]:
from autogen_agentchat.ui import Console

await Console(team.run_stream())

---------- TextMessage (add_1_agent_first) ----------
1
---------- TextMessage (add_1_agent_second) ----------
2
---------- TextMessage (add_1_agent_third) ----------
3
---------- TextMessage (add_1_agent_first) ----------
3
---------- TextMessage (add_1_agent_second) ----------
4
---------- TextMessage (add_1_agent_third) ----------
5
---------- TextMessage (add_1_agent_first) ----------
5
---------- TextMessage (add_1_agent_second) ----------
6
---------- TextMessage (add_1_agent_third) ----------
7
---------- TextMessage (add_1_agent_first) ----------
7
---------- TextMessage (add_1_agent_second) ----------
8
---------- TextMessage (add_1_agent_third) ----------
9
---------- TextMessage (add_1_agent_first) ----------
9


Error processing publish message for add_1_agent_second_8f178a3a-5927-4230-909f-87c8e459ff1c/8f178a3a-5927-4230-909f-87c8e459ff1c
Traceback (most recent call last):
  File "C:\Users\DELL\AppData\Roaming\Python\Python314\site-packages\autogen_core\_single_threaded_agent_runtime.py", line 606, in _on_message
    return await agent.on_message(
           ^^^^^^^^^^^^^^^^^^^^^^^
    ...<2 lines>...
    )
    ^
  File "C:\Users\DELL\AppData\Roaming\Python\Python314\site-packages\autogen_core\_base_agent.py", line 119, in on_message
    return await self.on_message_impl(message, ctx)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\DELL\AppData\Roaming\Python\Python314\site-packages\autogen_agentchat\teams\_group_chat\_sequential_routed_agent.py", line 67, in on_message_impl
    return await super().on_message_impl(message, ctx)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\DELL\AppData\Roaming\Python\Python314\site-packages\autogen_core\_routed_

CancelledError: 

In [8]:
from autogen_agentchat.conditions import MaxMessageTermination

max_termination = MaxMessageTermination(5)

In [12]:
team = RoundRobinGroupChat(
    [add_1_agent_first, add_1_agent_second, add_1_agent_third],
    termination_condition =max_termination
)

In [13]:
from autogen_agentchat.ui import Console

await Console(team.run_stream())

---------- TextMessage (add_1_agent_first) ----------
1
---------- TextMessage (add_1_agent_second) ----------
2
---------- TextMessage (add_1_agent_third) ----------
3
---------- TextMessage (add_1_agent_first) ----------
4
---------- TextMessage (add_1_agent_second) ----------
5


TaskResult(messages=[TextMessage(id='c17dbf31-51be-4184-8eab-2417c299ec43', source='add_1_agent_first', models_usage=RequestUsage(prompt_tokens=24, completion_tokens=1), metadata={}, created_at=datetime.datetime(2026, 3, 1, 7, 5, 1, 192202, tzinfo=datetime.timezone.utc), content='1', type='TextMessage'), TextMessage(id='fc0703c7-20b9-4af4-a370-4c61509032ce', source='add_1_agent_second', models_usage=RequestUsage(prompt_tokens=29, completion_tokens=1), metadata={}, created_at=datetime.datetime(2026, 3, 1, 7, 5, 2, 540167, tzinfo=datetime.timezone.utc), content='2', type='TextMessage'), TextMessage(id='900aba81-afc1-4156-a765-a67cc4bb454b', source='add_1_agent_third', models_usage=RequestUsage(prompt_tokens=39, completion_tokens=1), metadata={}, created_at=datetime.datetime(2026, 3, 1, 7, 5, 3, 262734, tzinfo=datetime.timezone.utc), content='3', type='TextMessage'), TextMessage(id='f0f0c1a0-14e1-49b8-a86a-a517c0081585', source='add_1_agent_first', models_usage=RequestUsage(prompt_tokens=

In [14]:
from autogen_agentchat.agents import AssistantAgent

In [4]:

agent1 = AssistantAgent(
    name = 'story_writer',
    model_client=model_client,
    system_message="Give me a vague story about a brave knight, keep it short no more than 40 words. if critic say 'THE END' anywhere. Only output 'THE END'"
)

agent2 = AssistantAgent(
    name = 'story_critic',
    model_client=model_client,
    system_message="Continue the story and critic it with feedback so that it includes a plot on spiderman, if there is no plot on spiderman. Keep it short and no more than 40 words. If it feels there is a plot on spiderman and complete, just say 'THE END'. Only output 'THE END' Ensure you provide the directions otherwie"
)

In [9]:
from autogen_agentchat.conditions import TextMentionTermination


text_mention_termination = TextMentionTermination('THE END')
teamWithTextTermination = RoundRobinGroupChat(
    [agent1, agent2],
    termination_condition=text_mention_termination
)

In [10]:
from autogen_agentchat.ui import Console

await Console(teamWithTextTermination.run_stream(task = 'Write a story about a brave knight.'))

---------- TextMessage (user) ----------
Write a story about a brave knight.
---------- TextMessage (story_writer) ----------
A brave knight ventured to a dragon's lair, armed with nothing but resolve and a legendary shield. Facing fiery trials and echoing roars, he stood unwavering, his courage extinguishing the dragon's wrath, restoring peace to the land.
---------- TextMessage (story_critic) ----------
Incorporate Spiderman into the tale by having him assist the knight in overcoming the dragon, merging their worlds and showcasing a blend of medieval and superhero valor.
---------- TextMessage (story_writer) ----------
THE END


TaskResult(messages=[TextMessage(id='42356781-d2db-48d4-a010-95b7d9929531', source='user', models_usage=None, metadata={}, created_at=datetime.datetime(2026, 6, 27, 3, 17, 4, 181277, tzinfo=datetime.timezone.utc), content='Write a story about a brave knight.', type='TextMessage'), TextMessage(id='e386128a-ed1e-483c-8e17-8358443ad307', source='story_writer', models_usage=RequestUsage(prompt_tokens=168, completion_tokens=49), metadata={}, created_at=datetime.datetime(2026, 6, 27, 3, 17, 5, 821928, tzinfo=datetime.timezone.utc), content="A brave knight ventured to a dragon's lair, armed with nothing but resolve and a legendary shield. Facing fiery trials and echoing roars, he stood unwavering, his courage extinguishing the dragon's wrath, restoring peace to the land.", type='TextMessage'), TextMessage(id='4627907a-c8fe-4518-a24c-cc1b6b73c32c', source='story_critic', models_usage=RequestUsage(prompt_tokens=252, completion_tokens=32), metadata={}, created_at=datetime.datetime(2026, 6, 27, 3

# Combining Termination Conditions

In [11]:
combined_termination = MaxMessageTermination(1) | TextMentionTermination('THE END')

from autogen_agentchat.conditions import TextMentionTermination


text_mention_termination = TextMentionTermination('THE END')
teamWithTextTermination = RoundRobinGroupChat(
    [agent1, agent2],
    termination_condition=combined_termination
)

In [12]:
from autogen_agentchat.ui import Console

await Console(teamWithTextTermination.run_stream(task = 'Write a story about a brave knight.'))

---------- TextMessage (user) ----------
Write a story about a brave knight.


TaskResult(messages=[TextMessage(id='e9d1b00c-a796-451d-8739-de116c55818e', source='user', models_usage=None, metadata={}, created_at=datetime.datetime(2026, 6, 27, 3, 17, 15, 90381, tzinfo=datetime.timezone.utc), content='Write a story about a brave knight.', type='TextMessage')], stop_reason='Maximum number of messages 1 reached, current message count: 1')

# External Termination

ExternalTermination: Enables programmatic control of termination from outside the run. This is useful for UI integration (e.g., “Stop” buttons in chat interfaces).

In [13]:
import asyncio
from autogen_ext.models.openai import OpenAIChatCompletionClient
from dotenv import load_dotenv
import os

load_dotenv()
api_key = os.getenv('OPENAI_API_KEY')
model_client = OpenAIChatCompletionClient(model='gpt-4o', api_key=api_key)

In [14]:
from autogen_agentchat.agents import AssistantAgent

agent1 = AssistantAgent(
    name = 'story_writer',
    model_client=model_client,
    system_message="Give the story about a brave knight, keep it short no more than 40 words. if critic say 'THE END' anywhere. Only output 'THE END'"
)

agent2 = AssistantAgent(
    name = 'story_critic',
    model_client=model_client,
    system_message="Continue the story and critic it with feedback. Keep it short and no more than 40 words. If it feels complete, just say 'THE END'. Only output 'THE END'"
)

from autogen_agentchat.conditions import ExternalTermination
# if button_click:
external_termination = ExternalTermination()

from autogen_agentchat.teams import RoundRobinGroupChat
team = RoundRobinGroupChat(
    [agent1, agent2],
    termination_condition= external_termination
)



In [15]:
from autogen_agentchat.ui import Console
run = asyncio.create_task(Console(team.run_stream(task = 'Write a story about a brave knight less than 40 words.')))

await asyncio.sleep(3)
external_termination.set()
await run

---------- TextMessage (user) ----------
Write a story about a brave knight less than 40 words.
---------- TextMessage (story_writer) ----------
A brave knight faced a fierce dragon, standing tall and unyielding. With courage blazing, he slayed the beast, saving the kingdom. Villagers hailed him, forever remembering his bravery and unwavering spirit.
---------- TextMessage (story_critic) ----------
THE END
---------- TextMessage (story_writer) ----------
THE END
---------- TextMessage (story_critic) ----------
THE END


TaskResult(messages=[TextMessage(id='8aeee49c-c41f-462a-b0a1-e8c639cd2904', source='user', models_usage=None, metadata={}, created_at=datetime.datetime(2026, 6, 27, 3, 17, 42, 703083, tzinfo=datetime.timezone.utc), content='Write a story about a brave knight less than 40 words.', type='TextMessage'), TextMessage(id='2419c477-9df6-4660-82e0-4840221fc00c', source='story_writer', models_usage=RequestUsage(prompt_tokens=58, completion_tokens=42), metadata={}, created_at=datetime.datetime(2026, 6, 27, 3, 17, 44, 332601, tzinfo=datetime.timezone.utc), content='A brave knight faced a fierce dragon, standing tall and unyielding. With courage blazing, he slayed the beast, saving the kingdom. Villagers hailed him, forever remembering his bravery and unwavering spirit.', type='TextMessage'), TextMessage(id='8cd669f0-ec86-4f83-bcfc-6589a46bac4e', source='story_critic', models_usage=RequestUsage(prompt_tokens=111, completion_tokens=2), metadata={}, created_at=datetime.datetime(2026, 6, 27, 3, 17, 4

In [ ]:
# The team is not stopping immediately but rather current agent is completing its run.

# Aborting A Team

Different from stopping a team, aborting a team will immediately stop the team and raise a CancelledError exception.

In [41]:
from autogen_core import CancellationToken

cancellation_token = CancellationToken()

run2 = asyncio.create_task(
    Console(team.run_stream(task = 'Give a short Story about a lion atmost 40 words',cancellation_token=cancellation_token))
)

await asyncio.sleep(5)
cancellation_token.cancel()

try:
    result = await run2
except :
    print("Task Was Cancelled")

---------- TextMessage (user) ----------
Give a short Story about a lion atmost 40 words
---------- TextMessage (story_writer) ----------
A wise lion ruled the savannah with fairness and strength. When a drought threatened, he led the animals to a distant oasis, ensuring their survival. His leadership inspired unity and harmony, securing peace and prosperity for the entire kingdom.
---------- TextMessage (story_critic) ----------
THE END
---------- TextMessage (story_writer) ----------
THE END
---------- TextMessage (story_critic) ----------
THE END
---------- TextMessage (story_writer) ----------
THE END
---------- TextMessage (story_critic) ----------
THE END


Error processing publish message for story_writer_5ddf632f-83fa-4624-86c0-6dfeca989cb9/5ddf632f-83fa-4624-86c0-6dfeca989cb9
Traceback (most recent call last):
  File "C:\Users\DELL\AppData\Roaming\Python\Python314\site-packages\autogen_core\_single_threaded_agent_runtime.py", line 606, in _on_message
    return await agent.on_message(
           ^^^^^^^^^^^^^^^^^^^^^^^
    ...<2 lines>...
    )
    ^
  File "C:\Users\DELL\AppData\Roaming\Python\Python314\site-packages\autogen_core\_base_agent.py", line 119, in on_message
    return await self.on_message_impl(message, ctx)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\DELL\AppData\Roaming\Python\Python314\site-packages\autogen_agentchat\teams\_group_chat\_sequential_routed_agent.py", line 67, in on_message_impl
    return await super().on_message_impl(message, ctx)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\DELL\AppData\Roaming\Python\Python314\site-packages\autogen_core\_routed_agent.

Task Was Cancelled
